# Сегментация океанических вихрей

Установите пакет в активное ядро и отредактируйте YAML. Новый пакет `0.1.0rc1` не заменяет стабильный скрипт в `legacy/`. Выполняйте тяжёлые ячейки только после проверки путей и настроек.

In [ ]:
from pathlib import Path
import sys
import subprocess

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
assert (ROOT / "pyproject.toml").exists(), "Укажите ROOT — путь к корню репозитория"
print("Python:", sys.executable)
print("Repository:", ROOT)

def run(*args):
    subprocess.run([sys.executable, "-m", "ocean_eddy", *map(str, args)], cwd=ROOT, check=True)

При необходимости установите пакет через `%pip install -e "/полный/путь/к/репозиторию[all]"` и перезапустите ядро. Для первоначальной проверки оставьте `num_workers: 0`.

In [ ]:
run("doctor")

In [ ]:
CONFIG = "configs/multiclass_deeplab.yaml"
# Перед запуском отредактируйте images_dir, masks_dir, output_dir, class_values и class_names.
print((ROOT / CONFIG).read_text(encoding="utf-8"))

In [ ]:
run("train", "--config", CONFIG)

In [ ]:
CHECKPOINT = "runs/deeplab_multiclass/best_checkpoint.pt"
TEST_IMAGES = "data/test/images"
TEST_MASKS = "data/test/masks"
PREDICTIONS = "predictions/multiclass"
run("infer", "--input", TEST_IMAGES, "--checkpoint", CHECKPOINT,
    "--output_dir", PREDICTIONS, "--stride", 256,
    "--blend_mode", "distance", "--smooth_sigma", 1.0,
    "--min_object_size", 200, "--save_prob")

In [ ]:
run("validate", "--pred_dir", PREDICTIONS, "--mask_dir", TEST_MASKS,
    "--images_dir", TEST_IMAGES, "--checkpoint", CHECKPOINT,
    "--output_dir", "metrics/multiclass")

Не настраивайте пороги на итоговом test. Предобработку HistEq/CLAHE применяйте одинаково к train/validation/test. Для её выходов задайте `zero_is_nodata: false`. Flip-аугментация/TTA могут менять смысл классов, связанных с закрученностью; в многоклассовом примере отражения выключены.